# LabLog YOLOv8 Fine-tuning (Colab)

실험 기구 객체 인식을 위해 `yolov8n.pt`를 fine-tuning한다.

**사전 준비:**
1. Google 계정으로 Colab 접속
2. **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU** 선택
3. Roboflow 계정 (https://app.roboflow.com) 생성 + API key 발급
4. Roboflow에 데이터셋 준비 (Universe fork 또는 본인 라벨링)

**출력:**
- `runs/lablog_yolo/weights/best.pt` — 학습된 가중치 (이 파일을 `backend/`에 복사)

## 1. 의존성 설치 + GPU 확인

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU가 활성화되지 않았습니다. 런타임 유형을 T4 GPU로 변경하세요.")

## 2. Roboflow에서 데이터셋 다운로드

Roboflow 사용 방법:
1. https://app.roboflow.com 로그인
2. **공개 데이터셋 사용**: https://universe.roboflow.com 에서 "lab equipment"/"chemistry lab" 검색 → 적당한 데이터셋 페이지 → **Fork Dataset**
3. **본인 데이터 사용**: 새 프로젝트 만들기 → `extract_frames.py`로 추출한 이미지 업로드 → Annotate 탭에서 bounding box 라벨링 → Generate → Version 생성
4. 데이터셋 페이지 → **Versions → Get Snippet → YOLOv8** → 코드 복사해 아래 셀의 API key/workspace/project/version 교체

API key는 Settings → API Key에서 확인.

In [ ]:
from roboflow import Roboflow

# TODO: 본인의 Roboflow snippet으로 교체 (Roboflow 데이터셋 페이지의 "Get Snippet" 참조)
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)
dataset = version.download("yolov8")

print(f"\n데이터셋 경로: {dataset.location}")
print(f"data.yaml: {dataset.location}/data.yaml")

In [ ]:
# data.yaml 내용 확인 (클래스 이름 + train/val 경로)
with open(f"{dataset.location}/data.yaml") as f:
    print(f.read())

## 3. 학습 (Fine-tuning)

**파라미터 가이드:**
- `epochs=50`: 데이터셋 크기에 따라 조절. 100장 미만이면 50, 500장 이상이면 100~200
- `imgsz=640`: 기본. 작은 객체가 많으면 1280으로 올려도 됨 (학습 느려짐)
- `batch=16`: T4에서 안전. OOM 발생 시 8로 낮춤
- `patience=10`: 10 epoch 동안 mAP 안 오르면 조기 종료 (시간 절약)
- 라벨이 적은 클래스는 `yolov8n.pt` 대신 `yolov8s.pt`(small)로 시작하면 정확도 향상 — 학습 시간 ~2배

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # 작은 모델로 시작 (Colab T4에서 빠름)

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    project="runs",
    name="lablog_yolo",
    exist_ok=True,
)

## 4. 검증 (validation set 성능 확인)

- **mAP50** > 0.7: 실용 가능
- **mAP50** 0.4~0.7: 데이터 추가 라벨링 후 재학습 권장
- **mAP50** < 0.4: 라벨 품질/양 점검 필요

In [ ]:
metrics = model.val()
print(f"\nmAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"\n클래스별 mAP50:")
for i, name in enumerate(model.names.values()):
    print(f"  {name}: {metrics.box.maps[i]:.3f}")

In [ ]:
# 예측 샘플 — 검증셋 첫 4장에 대한 예측 시각화 (선택)
import os
val_imgs = os.listdir(f"{dataset.location}/valid/images")[:4]
for img_name in val_imgs:
    img_path = f"{dataset.location}/valid/images/{img_name}"
    model.predict(img_path, save=True, project="runs", name="preview", exist_ok=True)
print("예측 이미지: runs/preview/ 폴더 확인")

## 5. `best.pt` 다운로드

다운로드된 `best.pt`를 로컬의 `backend/` 폴더에 복사한 후 통합하면 된다.

In [ ]:
from google.colab import files

best_path = "runs/lablog_yolo/weights/best.pt"
files.download(best_path)
print(f"다운로드: {best_path}")

## 6. 로컬 통합 절차 (이 셀은 실행 X — 가이드)

1. 다운로드한 `best.pt`를 `c:\Coding\lablog\backend\` 폴더에 배치
2. [`backend/analyzer.py`](analyzer.py)의 `DEFAULT_YOLO_WEIGHTS`를 `"best.pt"`로 변경
3. 학습 데이터셋의 클래스 목록(data.yaml의 `names`)을 확인
4. [`backend/vectorizer.py`](vectorizer.py)의 `YOLO_VOCAB`을 새 클래스 목록과 매핑해 갱신
   - **중요**: 순서 변경 시 GRU 가중치 호환 불가 → 5번 필수
5. GRU 재학습:
   ```powershell
   .\.venv\Scripts\python.exe train_gru.py
   ```
6. uvicorn 재시작 → 새 모델로 추론 확인